In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N,1)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print("Tensor shapes:")
print(X_train_tensor.shape, y_train_tensor.shape)
print(X_test_tensor.shape, y_test_tensor.shape)


In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 4. Print shape of one batch

images_batch, ages_batch = next(iter(train_loader))
print("Batch image shape:", images_batch.shape)
print("Batch age shape:", ages_batch.shape)

In [ ]:
# 5. Display sample images
# ex 5 pics
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    img = images_batch[i].permute(1, 2, 0).numpy()
    age = ages_batch[i].item()

    axes[i].imshow(img)
    axes[i].set_title(f"Age: {age}")
    axes[i].axis("off")

plt.show()

In [ ]:
# Task 1: Write your model class here:
import torch
import torch.nn as nn

class AgeRegressor4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim):
        super(AgeRegressor4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten image

        z1 = self.layer1(x)
        a1 = self.relu(z1)

        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        output = self.layer4(a3)
        return output


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, ages in loader:
        images, ages = images.to(device), ages.to(device)

        preds = model(images)
        loss = loss_fn(preds, ages)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, ages in loader:
            images, ages = images.to(device), ages.to(device)

            preds = model(images)
            loss = loss_fn(preds, ages)

            running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

C, H, W = X_train.shape[1], X_train.shape[2], X_train.shape[3]
input_dim = C * H * W

model = AgeRegressor4Layer(input_dim=input_dim, hidden_dim=512).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss Over Epochs")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
import numpy as np

model.eval()

images, true_ages = next(iter(test_loader))
images, true_ages = images.to(device), true_ages.to(device)

with torch.no_grad():
    preds = model(images).cpu().numpy().flatten()

true_ages = true_ages.cpu().numpy().flatten()

# just plot tha first 6 predictions
plt.figure(figsize=(15, 6))

for i in range(6):
    img = images[i].cpu().permute(1, 2, 0).numpy()

    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Pred: {preds[i]:.1f} | Actual: {true_ages[i]:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()
